# 🎯 BBS Customer & Marketing Analytics
## Team Assignment 2 — 2026
### BrandPulse Analytics: Decoding Three Brands

---

**Team Members:**
Giulia Lorelli, Francesco Gambera, Romain Derguini, Israa Ismail, Iolanda Costa 
**Submission Date:** Monday 11 May 2026
**Programme:** DSBA — Bologna Business School

---

## 📋 Question 0 — Data Loading & Preparation


In [ ]:
!pip install TextBlob emoji

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/Team Assignment 2 Data 2026.csv", encoding='utf-8-sig')
print(len(df))


In [ ]:
print(f"\n{'='*40}")
print(f"  DATASET OVERVIEW")
print(f"{'='*40}")
print(f"  Rows     : {len(df)}")
print(f"  Columns  : {len(df.columns)}")
print(f"  Nulls    : {df.isna().sum().sum()}")
print(f"{'='*40}\n")


In [ ]:
print(df.describe())
print(df.info())

In [ ]:
print(f"\n{'='*40}")
print("BRAND DISTRIBUTION")
print(df["Brand"].value_counts())
print("=" * 40)

In [ ]:
df["Post_Text"].head()

## 💬 Question 1 — Sentiment Analysis


Question 1 Sentiment Analysis - How Positive Are the Posts?
Calculate a sentiment polarity score (or code-scheme) for each of the 600 posts in your dataset.





In [ ]:
from textblob import TextBlob, Word, Blobber

def get_sentiment(text):
  return TextBlob(text).sentiment.polarity

df['Sentiment_Score'] = df['Post_Text'].apply(get_sentiment)


In [ ]:
df['Sentiment_Score']

In [ ]:
import matplotlib.pyplot as plt

df['Sentiment_Score'].hist(bins = 20)
plt.xlabel('Sentiment Score')
plt.ylabel("Frequency")

plt.title("Histogram for sentiment posts")

plt.show()



**What the Histogram shows:**


*   Most posts **cluster around 0 to 0.25 — slightly positive on average**
*  There are some genuinely negative posts (**around -0.75) but they're rare**
* The distribution is right-skewed — brands tend to post positively, which makes sense for marketing content


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, brand in zip(axes, df['Brand'].unique()):
    df[df['Brand'] == brand]['Sentiment_Score'].hist(bins=15, ax=ax)
    ax.set_title(brand)
    ax.set_xlabel('Sentiment Score')
    ax.set_ylabel('Frequency')

plt.tight_layout()
plt.show()



#### Sentiment Distribution by Brand

- **LumoraBeauty** shows a bell curve skewed slightly to the right, indicating
  generally positive sentiment with some variation. It has the widest distribution,
  reflecting a diverse range of post tones.

- **FitPulse** also shows a right-skewed bell curve, suggesting consistently
  positive and motivational content, which aligns with its energetic brand tone.

- **TechNest** shows a bimodal distribution with two peaks — one near neutral (0)
  and one slightly positive (0.2-0.4). This suggests TechNest alternates between
 factual posts and engaging content.

---



In [ ]:
df['Sentiment_Score'].groupby(df['Brand']).mean().plot(kind="bar")

plt.xlabel('Brand')
plt.ylabel("Avg. Setiment_Score")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.title("Bar chart for Average sentiment score by brand")
plt.show()
print(df.groupby('Brand')['Sentiment_Score'].mean().round(3))


In [ ]:
print(f"\n{'='*40}")
print("Bottom 3 (most negative) posts")
print(f"{'='*40}\n")
bottom3 = df.drop_duplicates('Post_Text').sort_values('Sentiment_Score').head(3)[['Post_Text', 'Brand', 'Sentiment_Score']]
for i, row in bottom3.iterrows():
  print(f"Brand: {row['Brand']} |  Score: {row['Sentiment_Score']:.2f} ")
  print(f"{row['Post_Text']}\n")


In [ ]:
print(f"\n{'='*40}")
print("Top 3 (most positive) posts")
print(f"{'='*40}\n")
top3 = df.drop_duplicates('Post_Text').sort_values('Sentiment_Score').tail(3)[['Post_Text', 'Brand', 'Sentiment_Score']]
for i, row in top3.iterrows():
    print(f"Brand: {row['Brand']} | Score: {row['Sentiment_Score']:.2f}")
    print(f"{row['Post_Text']}\n")

#### Question 1: Discussion
Though TextBlob enables fast sentiment classification, it has clear limitations.
* It **fails to detect irony or sarcasm,** which is common in social media posts.
* It also does **not interpret emojis**, which are a key part of brand communication
and often carry strong emotional signals.
* Finally, TextBlob is trained on general
text and may **not capture brand-specific language** or tone accurately.

## 🏷️ Question 2 — Topic Classification - What Are Brands Posting About?

Using a zero-shot classification pipeline from Hugging Face Transformers, classify each post into one of three intent categories: Informative, Entertaining, or Promotional.

* Build models to explain post engagement. Use Num_Likes and/or Num_Comments as your outcome variable(s). Choose at least one as your primary outcome.

*	Which derived NLP variables (sentiment, topic class, emoji count) predict engagement?

*	Do brand-level characteristics (follower count, posting frequency) matter?
*	Are there factors that drive likes but not comments, or vice versa?
Report your model results and draw marketing implications for BrandPulse and its clients.


In [ ]:
import datasets
from transformers import pipeline
from transformers.pipelines.pt_utils import KeyDataset
from tqdm.auto import tqdm

pipe = pipeline("zero-shot-classification",  model="facebook/bart-large-mnli")

text = df['Post_Text']

labels = ['Informative', 'Entertaining', 'Promotional']

result = pipe(text.tolist(), candidate_labels =labels)


In [ ]:
df['Topic_Class'] = [result['labels'][0] for result in result]

In [ ]:

order = df.groupby(['Brand', 'Topic_Class']).size().unstack().sum().sort_values(ascending=False).index.tolist()
ax = df.groupby(['Brand', 'Topic_Class']).size().unstack()[order].plot(
    kind='bar',
    color=['#FF6B6B', '#4ECDC4', '#45B7D1'],
    edgecolor='white'
)
plt.xlabel('Brand')
plt.ylabel('Number of Posts')
plt.title('Topic Classification by Brand')
plt.xticks(rotation=0)
plt.legend(title='Topic Class')
plt.grid(axis='y', linestyle='--', alpha=0.7)
ax.set_axisbelow(True)

plt.show()
print(df.groupby('Brand')['Topic_Class'].value_counts(normalize=True).round(3) * 100)



#### Question 2: Discussion


Informative content dominates across all three brands, suggesting that social media
posts in this dataset are primarily educational or value-driven rather than promotional.

- **LumoraBeauty** has the highest share of Informative posts (85.5%), with minimal
  Entertaining (7.0%) and Promotional (7.5%) content.
- **TechNest** follows with 77.0% Informative, 14.5% Entertaining, and 8.5% Promotional.
- **FitPulse** has the most balanced distribution — 70.5% Informative, 16.0% Entertaining,
  and 13.5% Promotional.

It is worth noting that the high proportion of Informative posts across all brands may
partly reflect a limitation of the zero-shot classification model, which may default
toward Informative when post intent is ambiguous.

### After

In [ ]:
for (brand, topic), group in df.groupby(['Brand', 'Topic_Class']):
    print(f"\nBrand: {brand} | Topic: {topic}")
    print(group['Post_Text'].sample(1).values[0])
    print("-" * 40)

#### Question 2: Post Examination

Overall, the zero-shot classifier performed reasonably well, correctly identifying
around 80% of posts based on manual inspection.

However, it struggled with subscription and newsletter posts — for example,
"TechNest newsletter subscribers receive early access to new product launches"
was classified as Informative when it is arguably Promotional, as it is driving
sign-ups. This suggests the model favours surface-level language (factual tone)
over underlying intent, which is a known limitation of zero-shot classification.

## 😊 Question 3: Paralinguistic / Emoji Analysis - The Role of Emojis

Extract and analyse emoji usage across all 600 posts using one of the following approaches:

*	Python 'emoji' library: use emoji.emoji_count(text) for counts and emoji.demojize(text) to get text descriptions of each emoji
*	VADER emoji extension: provides emoji-level sentiment scoring alongside text sentiment
           For each post, derive the following variables:
*	Emoji_Count: total number of emojis in the post
*	Emoji types: which emojis appear? Create a frequency table of common emojis per brand
*	Emoji sentiment (if your tool provides it): calculate valence score per post
*	Which brand uses the most emojis on average? Which uses the fewest?
*	Are certain emojis distinctive to one brand or one topic category?
*	Is there a correlation between emoji count and the sentiment score from Q1?



In [ ]:
print(df['Brand'].value_counts())

In [ ]:
import emoji

emoji_count = df['Post_Text'].apply(emoji.emoji_count)
emoji_desc = df['Post_Text'].apply(emoji.demojize)

df['Emoji_Count'] = emoji_count
df['Emoji_Desc'] = emoji_desc

print(emoji_count)

pd.set_option('display.max_colwidth', None)
print(emoji_desc.head(5))


In [ ]:
average_emoji_brand = df['Emoji_Count'].groupby(df['Brand']).mean()
print(average_emoji_brand)

#### Average Emoji Count by Brand

All three brands use a similar number of emojis per post, averaging between 2 and 3.

- **TechNest** leads with an average of **2.56 emojis per post**, which is surprising
  given the brand is described as using emojis selectively.
- **LumoraBeauty** follows closely with **2.44 emojis per post**.
- **FitPulse** uses the fewest on average at **2.30 emojis per post**.

Overall, the differences are small, suggesting emoji usage is relatively consistent
across all three brands.

In [ ]:
#emoji frequency table

import re

txt_emoji = df['Emoji_Desc']
x = txt_emoji.apply(lambda text: re.findall(r':[\w_]+:', text))



In [ ]:
flat_list =[]

for c in x:
  for s in c:
    flat_list.append(s)

pd.Series(flat_list).value_counts()

In [ ]:
import emoji

emoji_counts = pd.Series(flat_list).value_counts().head(10)
emoji_counts.index = emoji_counts.index.map(lambda x: emoji.emojize(x))
print(emoji_counts)
#pd.Series(flat_list).value_counts().head(10).index.map(lambda x: emoji.emojize(x))


#flat_list = []


#flat_list = []
#for l in x:
    #for v in l:  # ← use l here
        #flat_list.append(v)
#pd.Series(flat_list).value_counts()



In [ ]:
df['Emoji_List'] = x

for brand in df['Brand'].unique():
    print(f"\n{'='*40}")
    print(f"  {brand}")
    print(f"{'='*40}")
    brand_emojis = []
    for lst in df[df['Brand'] == brand]['Emoji_List']:
        for e in lst:
            brand_emojis.append(e)
    top = pd.Series(brand_emojis).value_counts().head(5)
    top.index = top.index.map(emoji.emojize)
    print(top)

#### Emoji Frequency by Brand

Emoji usage is highly distinctive across brands, reflecting each brand's tone and audience.

- **LumoraBeauty** favours soft, aesthetic emojis — ✨ (89), 💛 (53), 🌸 (46), 💕 (20), 🌙 (19) —
  consistent with its warm, aspirational beauty brand identity.

- **FitPulse** uses energetic, motivational emojis — 💪 (93), 🔥 (44), 👇 (42), 😂 (22), 😅 (20) —
  reflecting its active, community-driven fitness tone.

- **TechNest** relies on functional, informative emojis — 💻 (63), ⚡ (62), 👇 (51), 💡 (38), ✅ (28) —
  aligning with its informative and playful tech identity.

Overall, emoji choice is strongly aligned with brand personality, suggesting intentional
paralinguistic strategy across all three brands.

In [ ]:
print(df[['Emoji_Count', 'Sentiment_Score']].corr())

#### Correlation: Emoji Count vs Sentiment Score

The correlation between emoji count and sentiment score is nearly zero (r = 0.02),
suggesting that using more emojis does not necessarily make a post more positive in tone.
Brands appear to use emojis for emphasis and engagement rather than to signal sentiment.

In [ ]:
!pip install vaderSentiment


In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
analyzer = SentimentIntensityAnalyzer()
df['Emoji_Sentiment'] = df['Post_Text'].apply(lambda x: analyzer.polarity_scores(x)['compound'])
print(df['Emoji_Sentiment'].describe())
print(df.groupby('Brand')['Emoji_Sentiment'].mean().round(3))

#### Emoji Sentiment Analysis (VADER)

#### Emoji Frequency by Brand

Emoji usage is highly distinctive across brands, reflecting each brand's tone and audience.

- **LumoraBeauty** favours soft, aesthetic emojis — ✨ (89), 💛 (53), 🌸 (46), 💕 (20),
  🌙 (19) — consistent with its warm, aspirational beauty brand identity. The dominance
  of sparkles and floral emojis reinforces a luxury skincare image.

- **FitPulse** uses energetic, motivational emojis — 💪 (93), 🔥 (44), 👇 (42), 😂 (22),
  😅 (20) — reflecting its active, community-driven fitness tone. The flexed bicep is
  the single most used emoji across all brands.

- **TechNest** relies on functional, informative emojis — 💻 (63), ⚡ (62), 👇 (51),
  💡 (38) — aligning with its informative and playful tech identity. The pointing down
  emoji 👇 appears across FitPulse and TechNest, likely used to drive link clicks.

Overall, emoji choice is strongly aligned with brand personality, suggesting intentional
paralinguistic strategy across all three brands.